In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [4]:
loader = PyPDFLoader("../data/medical_report.pdf")
doc = loader.load()

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitter_data = splitter.split_documents(doc)

In [6]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [14]:
vector_store =  Chroma.from_documents(
    documents=splitter_data,
    embedding=embeddings,
    persist_directory="./vector_db"
)

#### Chain - Context_generator | prompt | llm | str-parser

In [15]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [16]:
def get_context(query: str):
    data = vector_store.similarity_search(query=query)

    context = ""
    for doc in data:
        context += doc.page_content + "\n"

    return {
        "context":context,
        "question":query
    }

In [17]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answer based on the context for user question. and 
    if you don't know the answer, then you can say that 'I dont know.'
    Context: {context}
    Question: {question}
""")

In [11]:
rag_chain = get_context | prompt | llm

In [12]:
res = rag_chain.invoke("Collected Date and time of this report? ")

In [13]:
print(res.content[0]["text"])

Based on the provided context, the collected date and time is **9/7/2025 4:56:00PM** (or **09/07/2025 04:56:00PM**).
